# AG_PRAXIS NB01 — Load and Inventory the CICIoMT2024 CSVs

Before I decide anything about models, windows, or splits, I need to know what is
actually in these files. The dataset paper describes the features, but I want to read
the columns myself rather than take the description on trust.

This notebook answers the questions I cannot proceed without. What are the columns and
what type is each one. Is there a timestamp anywhere. Is there anything that identifies
a sender or a receiver. Is the schema the same in every file or only in most of them.
How many rows are there per file and per attack family.

It writes two files. `data/processed/schema.json` records the column list, the dtypes,
and the answers to the timestamp and endpoint questions, so that no later notebook has
to guess at them. `config/feature_families.yaml` is a first grouping of the columns,
written as a draft because it is derived from spelling alone. Nothing is trained here.

The data sits on Drive and the code sits in the repository, so the first block mounts
one and clones the other. It also records the commit it is running from. Every number
printed below belongs to that commit, and if the working tree is dirty I would rather
see it now than when I am trying to explain a result later.

In [ ]:
import os
import subprocess
import sys
from datetime import date
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/AGREWAL14/AG_PRAXIS.git"

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/repo")
    if REPO_ROOT.exists():
        subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)
else:
    REPO_ROOT = Path.cwd()
    while not (REPO_ROOT / "config" / "base.yaml").exists() and REPO_ROOT != REPO_ROOT.parent:
        REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def git(*args):
    return subprocess.run(
        ["git", "-C", str(REPO_ROOT), *args], capture_output=True, text=True
    ).stdout.strip()


GIT_SHA = git("rev-parse", "--short", "HEAD")
GIT_BRANCH = git("rev-parse", "--abbrev-ref", "HEAD")
GIT_DIRTY = bool(git("status", "--porcelain"))
RUN_DATE = date.today().isoformat()

print(f"colab     : {IN_COLAB}")
print(f"repo root : {REPO_ROOT}")
print(f"git sha   : {GIT_SHA} on {GIT_BRANCH}" + ("   WORKING TREE DIRTY" if GIT_DIRTY else ""))
print(f"run date  : {RUN_DATE}")

Paths and the seed come from `config/base.yaml` rather than from anything typed into a
cell, so that changing a path is a commit and not an edit I forget I made. `FAST` is on
by default. In this notebook it controls one thing only: how many files are read in
full for the dtype comparison. The cheap checks, which are the column names and the row
counts, run over every file either way.

In [ ]:
import json
import random

import numpy as np
import pandas as pd

from src import inventory as inv

CFG = inv.load_config(REPO_ROOT)

SEED = CFG["seed"]
TRAIN_DIR = Path(CFG["paths"]["train_dir"])
TEST_DIR = Path(CFG["paths"]["test_dir"])
ARTIFACTS = Path(CFG["paths"]["artifacts"])

FAST = os.environ.get("FAST", "1") == "1"
FAST_FILE_LIMIT = 8

pd.set_option("display.max_rows", 250)
pd.set_option("display.width", 170)

print(f"seed      : {SEED}")
print(f"train dir : {TRAIN_DIR}   exists={TRAIN_DIR.exists()}")
print(f"test dir  : {TEST_DIR}   exists={TEST_DIR.exists()}")
print(f"artifacts : {ARTIFACTS}   exists={ARTIFACTS.exists()}")
print(f"FAST      : {int(FAST)}")
if FAST:
    print(f"            dtype check limited to the first {FAST_FILE_LIMIT} files")
    print("            a run entered in the ledger requires FAST=0")

No model is built here, but the seed is set before anything else runs so that any
sampling I do while looking at the data gives the same rows the next time.

In [ ]:
random.seed(SEED)
np.random.seed(SEED)
print(f"seeded with {SEED}")

First the files themselves. The class a file holds and the family it belongs to are both
readable from its name, so I derive them here and print the whole mapping. If the
derivation is wrong I would rather see it on one screen now than find it later inside a
count that does not add up.

In [ ]:
train_files = inv.list_csvs(TRAIN_DIR)
test_files = inv.list_csvs(TEST_DIR)
ALL_FILES = train_files + test_files
EXPECTED_FILES = 72

files = pd.DataFrame(
    {
        "file": [p.name for p in ALL_FILES],
        "split": [inv.split_name(p) for p in ALL_FILES],
        "class": [inv.class_name(p) for p in ALL_FILES],
        "family": [inv.family_name(p) for p in ALL_FILES],
    }
)

print(f"train files : {len(train_files)}")
print(f"test files  : {len(test_files)}")
print(f"total       : {len(ALL_FILES)}   (expected {EXPECTED_FILES})")
if len(ALL_FILES) != EXPECTED_FILES:
    print(f"MISMATCH: found {len(ALL_FILES)}, not {EXPECTED_FILES}. Check the paths printed above.")
print()
print(files.to_string(index=False))
print()
print(f"distinct classes  : {files['class'].nunique()}")
print(f"distinct families : {files['family'].nunique()}")
print()
print(files.groupby("family").size().sort_values(ascending=False).to_string())

Now the schema. I read one training file in full and print every column with its type,
how many nulls it has, how many distinct values it takes, and its range where it is
numeric. The distinct count is the part I care about most. A column holding one value
carries no information at all, and a column with as many distinct values as there are
rows is an identifier rather than a measurement, which would be a problem of a
different kind.

In [ ]:
REFERENCE_FILE = train_files[0]
ref = pd.read_csv(REFERENCE_FILE)

print(f"reference file : {REFERENCE_FILE.name}")
print(f"shape          : {ref.shape[0]:,} rows x {ref.shape[1]} columns")
print()

schema_table = inv.describe_columns(ref)
print(schema_table.to_string(index=False))

constant_cols = schema_table.loc[schema_table["n_unique"] <= 1, "column"].tolist()
rowwise_unique = schema_table.loc[schema_table["n_unique"] == len(ref), "column"].tolist()
null_cols = schema_table.loc[schema_table["nulls"] > 0, "column"].tolist()

print()
print(f"single-valued columns   : {constant_cols or 'none'}")
print(f"unique-per-row columns  : {rowwise_unique or 'none'}")
print(f"columns with any nulls  : {null_cols or 'none'}")

The first question that changes what I can build is whether there is a clock in the
data. I search the column names for time, timestamp, ts, date and epoch, and I do it two
ways. A whole-word match is a real hit. A letters-anywhere match is not, because `ts`
sits inside `Packets` and `ip` sits inside plenty of words, so those are listed
separately instead of being counted as findings.

In [ ]:
time_hits = inv.search_columns(ref.columns, inv.TIME_KEYWORDS)
HAS_TIMESTAMP = bool(time_hits["word_hits"])

print("searched", len(ref.columns), "column names for:", ", ".join(inv.TIME_KEYWORDS))
print()
for col, kws in time_hits["word_hits"].items():
    print(f"  whole word   : {col}   [{', '.join(kws)}]")
for col, kws in time_hits["substring_hits"].items():
    print(f"  letters only : {col}   [{', '.join(kws)}]   coincidence, not a field")
if not time_hits["word_hits"] and not time_hits["substring_hits"]:
    print("  nothing matched, on either reading")
print()
print("VERDICT   timestamp column present:", "YES" if HAS_TIMESTAMP else "NO")

The second question is whether anything identifies an endpoint. Same search, different
words: src, dst, ip, mac, addr, source, destination and port. This one decides whether
records can be grouped into flows or into a communication graph at all, or whether the
files are already reduced to per-record statistics with the addressing thrown away.

In [ ]:
endpoint_hits = inv.search_columns(ref.columns, inv.ENDPOINT_KEYWORDS)
HAS_ENDPOINTS = bool(endpoint_hits["word_hits"])

print("searched", len(ref.columns), "column names for:", ", ".join(inv.ENDPOINT_KEYWORDS))
print()
for col, kws in endpoint_hits["word_hits"].items():
    print(f"  whole word   : {col}   [{', '.join(kws)}]")
for col, kws in endpoint_hits["substring_hits"].items():
    print(f"  letters only : {col}   [{', '.join(kws)}]   coincidence, not a field")
if not endpoint_hits["word_hits"] and not endpoint_hits["substring_hits"]:
    print("  nothing matched, on either reading")
print()
print("VERDICT   endpoint identifiers present:", "YES" if HAS_ENDPOINTS else "NO")

Those two answers together decide how a sequence can be built, so the next block spells
that out rather than leaving me to remember it. This is also the check the decision log
records as outstanding, where the choice between sequence modelling and graph modelling
was left resting on whether endpoints exist.

In [ ]:
print("What this means for ordering records into sequences")
print()

if HAS_TIMESTAMP:
    print("A clock exists. Records can be sorted by time, gaps between them are")
    print("measurable, and a window can be cut on elapsed time rather than on row")
    print("position. Sort order must then be set explicitly and never assumed.")
else:
    print("There is no clock. The only ordering available is the order the rows appear")
    print("in the file, which is the order the feature extractor wrote them. That order")
    print("carries meaning inside a file and none across files, so a window must never")
    print("span two files, and the row index must never be shuffled before windowing.")

print()

if HAS_ENDPOINTS:
    print("Endpoint identifiers exist. Records can be grouped per host or per flow")
    print("before windowing, and nodes and edges for a communication graph are available.")
    print("They are also the most obvious shortcut in the data, so any model that reads")
    print("them has to be checked for learning the address instead of the behaviour.")
else:
    print("Nothing identifies a sender or a receiver. Records cannot be grouped into")
    print("flows or per-host streams, and there are no nodes from which to build a")
    print("communication graph. A sequence here is a run of consecutive rows from one")
    print("capture file and nothing finer than that.")

print()

if not HAS_TIMESTAMP and not HAS_ENDPOINTS:
    print("Both answers are negative, which fixes the definition used from here on:")
    print(f"a sequence is {CFG['sequence']['window']} consecutive rows taken from a single file at stride "
          f"{CFG['sequence']['stride']},")
    print("never crossing a file boundary. Graph modelling is not available on these")
    print("columns, because there is nothing to make a node out of.")

One file agreeing with itself proves nothing about the other seventy-one. Reading just
the header costs almost nothing, so I read the header of every file and compare the
column list against the reference exactly, order included. Any file that differs is
printed with what it is missing and what it has extra.

In [ ]:
REFERENCE_COLUMNS = list(ref.columns)
column_differences = []

for path in ALL_FILES:
    cols = inv.header_of(path)
    if cols == REFERENCE_COLUMNS:
        continue
    missing = [c for c in REFERENCE_COLUMNS if c not in cols]
    extra = [c for c in cols if c not in REFERENCE_COLUMNS]
    column_differences.append(
        {
            "file": path.name,
            "n_columns": len(cols),
            "missing": missing,
            "extra": extra,
            "same_columns_reordered": not missing and not extra,
        }
    )

SCHEMA_IDENTICAL = not column_differences

print(f"reference : {REFERENCE_FILE.name}, {len(REFERENCE_COLUMNS)} columns")
print(f"compared  : {len(ALL_FILES)} files")
print()
if SCHEMA_IDENTICAL:
    print("VERDICT   every file carries the same columns in the same order.")
else:
    print(f"VERDICT   {len(column_differences)} file(s) differ from the reference:")
    for d in column_differences:
        print(f"  {d['file']}   ({d['n_columns']} columns)")
        if d["missing"]:
            print(f"      missing : {d['missing']}")
        if d["extra"]:
            print(f"      extra   : {d['extra']}")
        if d["same_columns_reordered"]:
            print("      same columns, different order")

Matching column names do not mean matching types. pandas infers a type per file, so the
same column can come back as int64 in one file and float64 in another purely because one
of them happens to contain a missing value. That is a property of how the file was read
rather than of the data, so I check it separately and treat any difference as something
to look at rather than as a broken schema.

In [ ]:
dtype_files = ALL_FILES if not FAST else ALL_FILES[:FAST_FILE_LIMIT]
REFERENCE_DTYPES = {c: str(t) for c, t in ref.dtypes.items()}
dtype_differences = []

for path in dtype_files:
    for col, t in pd.read_csv(path).dtypes.items():
        reference_type = REFERENCE_DTYPES.get(col)
        if reference_type is not None and str(t) != reference_type:
            dtype_differences.append(
                {"file": path.name, "column": col, "reference": reference_type, "found": str(t)}
            )

print(f"read in full : {len(dtype_files)} of {len(ALL_FILES)} files" + ("   (FAST=1)" if FAST else ""))
print()
if not dtype_differences:
    print("No column changed type between the files read.")
else:
    print(f"{len(dtype_differences)} column/file pairs differ in inferred type:")
    print(pd.DataFrame(dtype_differences).to_string(index=False))
if FAST:
    print()
    print("This covers part of the corpus only. Rerun with FAST=0 before relying on it.")

Row counts come next, and they matter more than they look. Parsing seventy-two files
just to count lines is wasteful, so I count newlines instead, and check that shortcut
against pandas on the reference file before trusting it anywhere else.

In [ ]:
fast_count, pandas_count = inv.verify_row_count(REFERENCE_FILE)
print(f"newline count vs pandas on {REFERENCE_FILE.name} : {fast_count:,} vs {pandas_count:,}")
if fast_count == pandas_count:
    print("They agree, so newline counting is safe for the remaining files.")
else:
    print("They disagree, which means a field contains a newline. Counts below are wrong.")
print()

files["rows"] = [inv.count_data_rows(p) for p in ALL_FILES]
TOTAL_ROWS = int(files["rows"].sum())

print(files[["file", "split", "family", "rows"]].to_string(index=False))
print()

per_family = (
    files.groupby("family")["rows"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "rows", "count": "files"})
    .sort_values("rows", ascending=False)
)
per_family["share_pct"] = (100 * per_family["rows"] / TOTAL_ROWS).round(2)

print("rows per attack family:")
print(per_family.to_string())
print()
print(f"total rows across {len(ALL_FILES)} files : {TOTAL_ROWS:,}")
largest, smallest = per_family["rows"].iloc[0], per_family["rows"].iloc[-1]
print(
    f"largest family {per_family.index[0]} at {largest:,} rows, "
    f"smallest {per_family.index[-1]} at {smallest:,}, "
    f"a ratio of {largest / max(smallest, 1):,.0f} to 1"
)
print("That ratio is the reason macro-F1 is the primary metric and not accuracy.")

What the rest of the project needs from this notebook is the column list, the types, and
the two boolean answers. Those go to `data/processed/schema.json` so that no later
notebook has to re-derive them or, worse, assume them. The search results are written
alongside, because a bare `false` is not evidence of anything on its own.

In [ ]:
schema_doc = {
    "generated_by": "AG_PRAXIS_NB01_load_inventory.ipynb",
    "generated_on": RUN_DATE,
    "git_sha": GIT_SHA,
    "fast_mode": FAST,
    "reference_file": REFERENCE_FILE.name,
    "n_files": len(ALL_FILES),
    "n_columns": len(REFERENCE_COLUMNS),
    "total_rows": TOTAL_ROWS,
    "has_timestamp": HAS_TIMESTAMP,
    "has_endpoint_identifiers": HAS_ENDPOINTS,
    "schema_identical_across_files": SCHEMA_IDENTICAL,
    "files_differing_from_reference": [d["file"] for d in column_differences],
    "columns": REFERENCE_COLUMNS,
    "dtypes": REFERENCE_DTYPES,
    "timestamp_search": {"keywords": inv.TIME_KEYWORDS, **time_hits},
    "endpoint_search": {"keywords": inv.ENDPOINT_KEYWORDS, **endpoint_hits},
    "rows_per_file": dict(zip(files["file"], files["rows"].astype(int))),
    "rows_per_family": {k: int(v) for k, v in per_family["rows"].items()},
}

SCHEMA_PATH = REPO_ROOT / "data" / "processed" / "schema.json"
SCHEMA_PATH.parent.mkdir(parents=True, exist_ok=True)
SCHEMA_PATH.write_text(json.dumps(schema_doc, indent=2) + "\n")
print(f"wrote {SCHEMA_PATH}")

if ARTIFACTS.exists():
    drive_copy = ARTIFACTS / "NB01" / "schema.json"
    drive_copy.parent.mkdir(parents=True, exist_ok=True)
    drive_copy.write_text(json.dumps(schema_doc, indent=2) + "\n")
    print(f"copied to {drive_copy}")

print()
print(json.dumps({k: v for k, v in schema_doc.items() if k not in ("rows_per_file",)}, indent=2))

The last thing to write is a first grouping of the columns into timing, protocol,
statistical and other. This is guesswork from spelling and nothing else, so the file
records itself as a draft and lists every column that matched more than one rule. Rate
is the obvious problem. It reads as timing to me because it is a per-second quantity,
but it is computed from a count, and a column like `ack_count` matches both protocol and
statistical with equal justice. I would rather flag those than quietly decide them.

In [ ]:
assignment = inv.assign_families(REFERENCE_COLUMNS)

for name, cols in assignment["families"].items():
    print(f"{name:<12} {len(cols):>3}")
    for col in cols:
        print(f"             {col}")
print()
print(f"label columns : {assignment['labels'] or 'none found by name'}")
print(f"ambiguous     : {len(assignment['ambiguous'])} column(s) matched more than one rule")
for col, names in assignment["ambiguous"].items():
    print(f"                {col}  ->  {' / '.join(names)}  (assigned to {names[0]})")

families_yaml = inv.render_feature_families_yaml(
    assignment,
    source_file=REFERENCE_FILE.name,
    git_sha=GIT_SHA,
    run_date=RUN_DATE,
)

FAMILIES_PATH = REPO_ROOT / "config" / "feature_families.yaml"
FAMILIES_PATH.write_text(families_yaml)
print()
print(f"wrote {FAMILIES_PATH} as a draft")
print()
print(families_yaml)

The entry for `RESULTS_LEDGER.md`, ready to paste.

In [ ]:
status = "reference run" if not FAST else "EXPLORATORY, FAST=1, do not enter in the ledger"

ledger = f"""
### NB01 — data inventory ({RUN_DATE})

| field | value |
|---|---|
| notebook | AG_PRAXIS_NB01_load_inventory.ipynb |
| run date | {RUN_DATE} |
| git sha | {GIT_SHA}{" (working tree dirty)" if GIT_DIRTY else ""} |
| seed | {SEED} |
| FAST | {int(FAST)} |
| status | {status} |
| files read | {len(ALL_FILES)} of {EXPECTED_FILES} expected |
| columns | {len(REFERENCE_COLUMNS)} |
| total rows | {TOTAL_ROWS:,} |
| families | {files['family'].nunique()} |
| classes | {files['class'].nunique()} |
| timestamp column | {"yes" if HAS_TIMESTAMP else "no"} |
| endpoint identifiers | {"yes" if HAS_ENDPOINTS else "no"} |
| schema identical across files | {"yes" if SCHEMA_IDENTICAL else f"no, {len(column_differences)} file(s) differ"} |
| dtype differences | {len(dtype_differences)} across {len(dtype_files)} files read in full |
| metrics | none, no model trained |
| artefacts | data/processed/schema.json, config/feature_families.yaml (draft) |

Ordering available for sequences: {"timestamp" if HAS_TIMESTAMP else "row order within a file only"}.
Grouping available for sequences: {"endpoint identifiers present" if HAS_ENDPOINTS else "none, windows cannot cross a file boundary"}.
"""

print(ledger)